In [1]:
from omnivoice import OmniVoice
import torch

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16
)

print("OmniVoice loaded successfully.")

H:\conda\envs\omnivoice\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
H:\conda\envs\omnivoice\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
Loading weights: 100%|██████████| 527/527 [00:00<00:00, 1117.63it/s]

OmniVoice loaded successfully.


In [9]:
from pathlib import Path
import soundfile as sf
from IPython.display import Audio, display

ref_audio = Path(
    r"H:\Projects\speech_synthesis\speech_synthesis\samples\ALS\Real\Patient_48\001_Patient_48_10302014190442_recording.wav"
)

output_folder = Path(
    r"H:\Projects\speech_synthesis\speech_synthesis\samples\ALS\OmniVoice\Patient_48"
)

output_folder.mkdir(parents=True, exist_ok=True)

output_audio = output_folder / ref_audio.name

In [10]:
ref_text = (
    "I owe you a yoyo today. "
    "I owe you a yoyo today. "
    "I owe you a yoyo today. "
    "I owe you a yoyo today. "
    "I owe you a yoyo today."
)

target_text = (
    "I owe you a yoyo today. "
    "I owe you a yoyo today. "
    "I owe you a yoyo today. "
    "I owe you a yoyo today. "
    "I owe you a yoyo today."
)

In [11]:
audio = model.generate(
    text=target_text,
    ref_audio=str(ref_audio),
    ref_text=ref_text,
)

In [12]:
sf.write(
    output_audio,
    audio[0],
    24000
)

print("Saved to:")
print(output_audio)

Saved to:
H:\Projects\speech_synthesis\speech_synthesis\samples\ALS\OmniVoice\Patient_48\001_Patient_48_10302014190442_recording.wav


In [39]:
from pathlib import Path
import soundfile as sf
import torch
import gc


# ============================================================
# CHANGE ONLY THIS PATIENT FOLDER
# ============================================================

patient_name = "Patient_423"


# ============================================================
# PATHS
# ============================================================

input_folder = Path(
    rf"H:\Projects\speech_synthesis\speech_synthesis\samples\ALS\LFlow\{patient_name}"
)

output_folder = Path(
    rf"H:\Projects\speech_synthesis\speech_synthesis\samples\ALS\OmniVoice\{patient_name}"
)

output_folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# TEXT
# ============================================================

sentence = "I owe you a yoyo today."

ref_text = " ".join([sentence] * 5)
target_text = " ".join([sentence] * 5)


# ============================================================
# FIND RECORDINGS
# ============================================================

wav_files = sorted(input_folder.glob("*.wav"))

print(f"Patient: {patient_name}")
print(f"Found {len(wav_files)} recordings.")
print(f"Output: {output_folder}\n")


# ============================================================
# GENERATE
# ============================================================

generated = 0
skipped = 0
failed = []


for index, ref_audio in enumerate(wav_files, start=1):

    output_audio = output_folder / ref_audio.name

    # Skip files already generated
    if output_audio.exists():
        print(
            f"[{index}/{len(wav_files)}] "
            f"SKIP: {ref_audio.name}"
        )
        skipped += 1
        continue

    try:
        print(
            f"[{index}/{len(wav_files)}] "
            f"Generating: {ref_audio.name}"
        )

        audio = model.generate(
            text=target_text,
            ref_audio=str(ref_audio),
            ref_text=ref_text,
        )

        sf.write(
            output_audio,
            audio[0],
            24000
        )

        generated += 1

        # Release temporary memory
        del audio
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as error:

        failed.append(
            (ref_audio.name, str(error))
        )

        print(f"    ERROR: {error}")

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 60)
print(f"FINISHED: {patient_name}")
print("=" * 60)

print(f"Generated: {generated}")
print(f"Skipped:   {skipped}")
print(f"Failed:    {len(failed)}")

if failed:
    print("\nFailed files:")

    for filename, error in failed:
        print(f"\n{filename}")
        print(f"    {error}")

Patient: Patient_423
Found 5 recordings.
Output: H:\Projects\speech_synthesis\speech_synthesis\samples\ALS\OmniVoice\Patient_423

[1/5] Generating: 001_Patient_423.wav
[2/5] Generating: 003_Patient_423.wav


Reference audio is 20.8s long (>20s). This may cause slower generation, higher memory usage, and degraded voice cloning quality. We recommend trimming it to 3-10s.


[3/5] Generating: 008_Patient_423.wav
[4/5] Generating: 014_Patient_423.wav


Reference audio is 23.8s long (>20s). This may cause slower generation, higher memory usage, and degraded voice cloning quality. We recommend trimming it to 3-10s.


[5/5] Generating: 018_Patient_423.wav

FINISHED: Patient_423
Generated: 5
Skipped:   0
Failed:    0


In [50]:
from pathlib import Path
import soundfile as sf
import torch
import gc


# ============================================================
# CHOOSE 10 SPEAKERS
# ============================================================

patient_names = [
    "Patient_443",
    "Patient_458",
]


# ============================================================
# ROOT PATHS
# ============================================================

input_root = Path(
    r"H:\Projects\speech_synthesis\speech_synthesis\samples\ALS\LFlow"
)

output_root = Path(
    r"H:\Projects\speech_synthesis\speech_synthesis\samples\ALS\OmniVoice"
)

output_root.mkdir(parents=True, exist_ok=True)


# ============================================================
# TEXT
# ============================================================

sentence = "I owe you a yoyo today."

ref_text = " ".join([sentence] * 5)
target_text = " ".join([sentence] * 5)


# ============================================================
# COUNTERS
# ============================================================

total_generated = 0
total_skipped = 0
failed = []


# ============================================================
# PROCESS SPEAKERS SEQUENTIALLY
# ============================================================

for patient_index, patient_name in enumerate(
    patient_names,
    start=1
):

    input_folder = input_root / patient_name
    output_folder = output_root / patient_name

    if not input_folder.exists():
        print(
            f"\n[{patient_index}/{len(patient_names)}] "
            f"{patient_name}: folder not found, skipping."
        )
        continue

    output_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    wav_files = sorted(
        input_folder.glob("*.wav")
    )

    print("\n" + "=" * 70)
    print(
        f"[{patient_index}/{len(patient_names)}] "
        f"{patient_name}"
    )
    print(f"Recordings: {len(wav_files)}")
    print("=" * 70)

    patient_generated = 0
    patient_skipped = 0


    # --------------------------------------------------------
    # PROCESS EACH RECORDING
    # --------------------------------------------------------

    for wav_index, ref_audio in enumerate(
        wav_files,
        start=1
    ):

        output_audio = (
            output_folder / ref_audio.name
        )

        # Skip existing output
        if output_audio.exists():
            print(
                f"  [{wav_index}/{len(wav_files)}] "
                f"SKIP: {ref_audio.name}"
            )

            patient_skipped += 1
            total_skipped += 1
            continue


        try:

            print(
                f"  [{wav_index}/{len(wav_files)}] "
                f"Generating: {ref_audio.name}"
            )

            audio = model.generate(
                text=target_text,
                ref_audio=str(ref_audio),
                ref_text=ref_text,
            )

            sf.write(
                output_audio,
                audio[0],
                24000
            )

            patient_generated += 1
            total_generated += 1

            # Release temporary memory
            del audio

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()


        except torch.OutOfMemoryError as error:

            print(
                f"      CUDA OOM: {ref_audio.name}"
            )

            failed.append(
                (
                    patient_name,
                    ref_audio.name,
                    str(error)
                )
            )

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()


        except Exception as error:

            print(
                f"      ERROR: {error}"
            )

            failed.append(
                (
                    patient_name,
                    ref_audio.name,
                    str(error)
                )
            )

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()


    # --------------------------------------------------------
    # PATIENT SUMMARY
    # --------------------------------------------------------

    print(
        f"\n{patient_name} finished:"
        f" generated={patient_generated},"
        f" skipped={patient_skipped}"
    )

    # Extra cleanup between speakers
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("10-SPEAKER BATCH FINISHED")
print("=" * 70)

print(f"Generated: {total_generated}")
print(f"Skipped:   {total_skipped}")
print(f"Failed:    {len(failed)}")


if failed:

    print("\nFailed files:")

    for patient_name, filename, error in failed:

        print(
            f"\n{patient_name} / {filename}"
        )

        print(
            f"    {error}"
        )


[1/2] Patient_443
Recordings: 8
  [1/8] Generating: 001_Patient_443.wav
  [2/8] Generating: 004_Patient_443.wav
  [3/8] Generating: 010_Patient_443.wav
  [4/8] Generating: 016_Patient_443.wav
  [5/8] Generating: 025_Patient_443.wav
  [6/8] Generating: 028_Patient_443.wav
  [7/8] Generating: 039_Patient_443.wav
  [8/8] Generating: 042_Patient_443.wav

Patient_443 finished: generated=8, skipped=0

[2/2] Patient_458
Recordings: 8
  [1/8] Generating: 001_Patient_458.wav
  [2/8] Generating: 010_Patient_458.wav
  [3/8] Generating: 022_Patient_458.wav
  [4/8] Generating: 038_Patient_458.wav
  [5/8] Generating: 052_Patient_458.wav
  [6/8] Generating: 071_Patient_458.wav
  [7/8] Generating: 087_Patient_458.wav


Reference audio is 25.1s long (>20s). This may cause slower generation, higher memory usage, and degraded voice cloning quality. We recommend trimming it to 3-10s.


  [8/8] Generating: 127_Patient_458.wav

Patient_458 finished: generated=8, skipped=0

10-SPEAKER BATCH FINISHED
Generated: 16
Skipped:   0
Failed:    0
